In [ ]:
# --- IMPORTS ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
import os


print("--- STARTING SYSTEM ---")

# initialising the embedding model all-MiniLM-L6-v2 
print("1. Initializing embedding model...")
# converts every chunk of text into a vector
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Loading vector database 
print("2. Loading Course Content Index...")
if os.path.exists("faiss_index_react"):
    vector_db = FAISS.load_local("faiss_index_react", embedding_model, allow_dangerous_deserialization=True)
else:
    print("ERROR: Checkpoint not found!")

# Initialising llama 3.2 model running in the background Ollama app
# temperature=0 is for the llm to be strict and factual, and not be creative
print("3. Connecting to Llama 3.2...")
llm = OllamaLLM(model="llama3.2", temperature=0)

# extracting guidelines
# the guideline PDF is split into a list of individual items
print("4. Extracting Guidelines from PDF...")
loader = PyPDFLoader("guideline.pdf") 
guideline_pages = loader.load()

# Combining all the text and splitting it by new lines as part of extraction
# assuming the PDF has one guideline per line
full_text = "\n".join([page.page_content for page in guideline_pages])


# cleaning the guidelines
raw_lines = full_text.split('\n')
clean_guidelines = []

for line in raw_lines:
    line = line.strip()
    # Skipping empty or too short lines
    if len(line) < 10: 
        continue
    #  Skipping incomplete sentences 
    if line.endswith(";"): 
        continue
    # Skiping common headers
    if "students will be able to" in line.lower(): 
        continue
    
    clean_guidelines.append(line)

print(f"Found {len(clean_guidelines)} potential guidelines to check.")

#  defining the grading prompt
prompt_template = """
You are an strict academic assistant. Your task is to check if a specific Guideline is covered in the Course Content.

GUIDELINE: "{guideline}"

COURSE CONTENT FRAGMENTS: 
"{context}"

INSTRUCTIONS:
1. Determine if the guideline is "Fully Covered", "Partially Covered", or "Not Covered".
2. Provide a concise explanation.

FORMAT YOUR ANSWER EXACTLY LIKE THIS:
Match: [Fully Covered / Partially Covered / Not Covered]
Reasoning: [Your explanation here]
"""

prompt = PromptTemplate(input_variables=["guideline", "context"], template=prompt_template)

#  the loop
results_log = []

print("\n--- STARTING ANALYSIS LOOP ---\n")

for i, rule in enumerate(clean_guidelines): 
    print(f"Processing Guideline {i+1}...")
    
    # searching
    search_results = vector_db.similarity_search(rule, k=3)
    context_text = "\n".join([doc.page_content for doc in search_results])
    
    # reasining
    final_prompt = prompt.format(guideline=rule, context=context_text)
    response = llm.invoke(final_prompt)
    
    # printing and storing
    output = f"Guideline {i+1}: {rule}\n{response}\n{'-'*60}\n"
    print(output)
    results_log.append(output)

#saving the report as a txt file
with open("final_analysis_report.txt", "w", encoding="utf-8") as f:
    f.writelines(results_log)

print("Analysis Complete! Report saved to 'final_analysis_report.txt'")

--- STARTING SYSTEM ---
1. Initializing embedding model...
2. Loading Course Content Index...
3. Connecting to Llama 3.2...
4. Extracting Guidelines from PDF...
Found 4 potential guidelines to check.

--- STARTING ANALYSIS LOOP ---

Processing Guideline 1...
Guideline 1: PHP Conditional Statements
Match: Fully Covered
Reasoning: The guideline "PHP Conditional Statements" is fully covered in the Course Content Fragments. The content explains the if statement, if-else statement, if-elseif-else statement, and switch statement, providing a comprehensive overview of PHP's conditional statements.
------------------------------------------------------------

Processing Guideline 2...
Guideline 2: PHP Session variables
Match: Partially Covered
Reasoning: The guideline "PHP Session variables" is not explicitly mentioned in the provided Course Content Fragments. However, it can be inferred that session variables are used in PHP programming, as they are a fundamental concept in web development. T